In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

/home/pia/PycharmProjects/spatial_extraction_fairytales/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Exermine Roberta Structure

In [2]:
# model: https://huggingface.co/FacebookAI/xlm-roberta-large-finetuned-conll03-german
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large-finetuned-conll03-german")
model = AutoModelForTokenClassification.from_pretrained("FacebookAI/xlm-roberta-large-finetuned-conll03-german", device_map="auto")

print(tokenizer.cls_token) # <s>
print(tokenizer.sep_token) # </s>
print(tokenizer.mask_token) # <mask>
print(tokenizer.pad_token) # <pad>

Some weights of the model checkpoint at FacebookAI/xlm-roberta-large-finetuned-conll03-german were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


<s>
</s>
<mask>
<pad>


In [3]:
text = "Zur Zeit, wo das Wünschen noch geholfen hat, ward ein Königssohn von einer alten Hexe verwünscht, daß er im Walde in einem großen Eisenofen sitzen sollte. Da brachte er viele Jahre zu, und konnte ihn niemand erlösen."

tokens = tokenizer.tokenize(text)
print(tokens)

['▁Zur', '▁Zeit', ',', '▁wo', '▁das', '▁Wünsche', 'n', '▁noch', '▁geholfen', '▁hat', ',', '▁', 'ward', '▁ein', '▁König', 'sso', 'h', 'n', '▁von', '▁einer', '▁alten', '▁He', 'xe', '▁ver', 'w', 'ün', 's', 'cht', ',', '▁daß', '▁er', '▁im', '▁Wald', 'e', '▁in', '▁einem', '▁großen', '▁Eisen', 'of', 'en', '▁si', 'tzen', '▁sollte', '.', '▁Da', '▁brachte', '▁er', '▁viele', '▁Jahre', '▁zu', ',', '▁und', '▁konnte', '▁ihn', '▁niemand', '▁er', 'lös', 'en', '.']


In [4]:
input_ids = tokenizer(text)["input_ids"]
print(input_ids)

print(tokenizer.convert_ids_to_tokens(input_ids))

[0, 47426, 5502, 4, 4323, 381, 92659, 19, 1838, 226204, 1256, 4, 6, 19364, 599, 89537, 10798, 127, 19, 542, 2235, 83832, 1529, 4878, 493, 434, 7640, 7, 5204, 4, 44124, 72, 566, 59757, 13, 23, 2315, 30909, 138745, 4390, 33, 78, 2731, 12961, 5, 930, 201178, 72, 9510, 12241, 404, 4, 165, 25237, 17889, 46203, 72, 28739, 33, 5, 2]
['<s>', '▁Zur', '▁Zeit', ',', '▁wo', '▁das', '▁Wünsche', 'n', '▁noch', '▁geholfen', '▁hat', ',', '▁', 'ward', '▁ein', '▁König', 'sso', 'h', 'n', '▁von', '▁einer', '▁alten', '▁He', 'xe', '▁ver', 'w', 'ün', 's', 'cht', ',', '▁daß', '▁er', '▁im', '▁Wald', 'e', '▁in', '▁einem', '▁großen', '▁Eisen', 'of', 'en', '▁si', 'tzen', '▁sollte', '.', '▁Da', '▁brachte', '▁er', '▁viele', '▁Jahre', '▁zu', ',', '▁und', '▁konnte', '▁ihn', '▁niemand', '▁er', 'lös', 'en', '.', '</s>']


ID = 0 ->    < s > Token   (Start Eingabe)

ID = 2 ->    < /s > Token   (Ende Eingabe)

In [5]:
print(type(tokenizer))
print(type(tokenizer.backend_tokenizer))
print(tokenizer.backend_tokenizer.model)
print(type(tokenizer.backend_tokenizer.model))

<class 'transformers.models.xlm_roberta.tokenization_xlm_roberta_fast.XLMRobertaTokenizerFast'>
<class 'tokenizers.Tokenizer'>
Unigram(unk_id=3, vocab=[("<s>", 0), ("<pad>", 0), ("</s>", 0), ("<unk>", 0), (",", -3.4635426998138428), ...], byte_fallback=False)
<class 'tokenizers.models.Unigram'>


Model uses Sentence-piece Unigram tokenizer algorithm: https://huggingface.co/docs/transformers/main/en/tokenizer_summary#unigram


## Testing Fairytale

In [6]:
from dataset_preprocessing import load_dataset

No errors found.
Lengths Documents:
9200
1482.8
1222.5
Lengths Sentences:
120
25.733649981731823
24.0


In [7]:
DATA_PATH = "data/grimm_fairytales_annotated/fuzzy_loc_3.7.26.json"
dataset = load_dataset(DATA_PATH)

fairytale = next((ft for ft in dataset.fairytales if ft.khm == 1), None)

In [8]:
# Tokenizer with max 512 length chunks and stride

encoding = tokenizer(
    fairytale.text,
    max_length=512,
    truncation=True,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

In [9]:
# encoding.tokens()
# encoding["offset_mapping"]

### Checking if Offset Chunks still fit

In [10]:
# Count Chunks

print("Originaltext:", len(fairytale.text))
print("Anzahl Chunks:", len(encoding["input_ids"]))

for i, offsets in enumerate(encoding["offset_mapping"]):
    print(f"Chunk {i}: {len(offsets)} Tokens")

Originaltext: 7437
Anzahl Chunks: 5
Chunk 0: 512 Tokens
Chunk 1: 512 Tokens
Chunk 2: 512 Tokens
Chunk 3: 512 Tokens
Chunk 4: 493 Tokens


In [11]:
# Offset Chunks

for i, offsets in enumerate(encoding["offset_mapping"]):
    print(f"\n--- Chunk {i} ---")

    for token, offset in zip(
        tokenizer.convert_ids_to_tokens(encoding["input_ids"][i]),
        offsets
    ):
        print(f"{token:20} {offset}")


--- Chunk 0 ---
<s>                  (0, 0)
▁In                  (0, 2)
▁den                 (3, 6)
▁alten               (7, 12)
▁Zeiten              (13, 19)
,                    (19, 20)
▁wo                  (21, 23)
▁das                 (24, 27)
▁Wünsche             (28, 35)
n                    (35, 36)
▁noch                (37, 41)
▁geholfen            (42, 50)
▁hat                 (51, 54)
,                    (54, 55)
▁lebt                (56, 60)
e                    (60, 61)
▁ein                 (62, 65)
▁König               (66, 71)
,                    (71, 72)
▁dessen              (73, 79)
▁Tö                  (80, 82)
chter                (82, 87)
▁waren               (88, 93)
▁alle                (94, 98)
▁schön               (99, 104)
,                    (104, 105)
▁aber                (106, 110)
▁die                 (111, 114)
▁jüngst              (115, 121)
e                    (121, 122)
▁war                 (123, 126)
▁so                  (127, 129)
▁schön         

In [12]:
# Are Token-Offset and token fitting?

for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

    for start, end in offsets:

        if start == end:
            continue

        assert 0 <= start < end <= len(fairytale.text), (
            f"Invalid offset: {(start, end)}"
        )

In [13]:
# Look at Overlap Chunks

for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

    valid_offsets = [
        (start, end)
        for start, end in offsets
        if start != end
    ]

    chunk_start = min(start for start, end in valid_offsets)
    chunk_end = max(end for start, end in valid_offsets)

    print(
        f"Chunk {chunk_idx}: "
        f"characters {chunk_start}–{chunk_end}"
    )

Chunk 0: characters 0–1889
Chunk 1: characters 1444–3288
Chunk 2: characters 2809–4661
Chunk 3: characters 4210–6052
Chunk 4: characters 5592–7437


In [14]:
covered = set()

for offsets in encoding["offset_mapping"]:

    for start, end in offsets:

        if start == end:
            continue

        covered.update(range(start, end))

missing = [
    i
    for i in range(len(fairytale.text))
    if i not in covered
]

print("Nicht abgedeckte Zeichen:", len(missing))

Nicht abgedeckte Zeichen: 1325


-> Zeichen vermutlich whitespaces, Satzzeichen etc.

In [15]:
# Check Annotation in Token-Offset

def check_annotation_coverage(fairytale, encoding):
    for span in fairytale.resolve_nested_spans():

        covered_tokens = []

        for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

            for token_idx, (start, end) in enumerate(offsets):

                # ignore special Tokens
                if start == end:
                    continue

                # Token overlaps annotation
                if start < span.end and end > span.start:
                    covered_tokens.append(
                        (chunk_idx, token_idx, start, end)
                    )

        covered_start = min(start for _, _, start, _ in covered_tokens)
        covered_end = max(end for _, _, _, end in covered_tokens)

        if covered_start > span.start or covered_end < span.end:
            print(
                f"NICHT vollständig abgedeckt:\n"
                f"  Span: {span.text!r}\n"
                f"  Offset: ({span.start}, {span.end})\n"
                f"  Token-Abdeckung: ({covered_start}, {covered_end})"
            )

In [16]:
check_annotation_coverage(
    fairytale,
    encoding
)

In [17]:
def check_uncovered_chars(text, offsets):
    covered = set()

    for chunk_offsets in offsets:
        for start, end in chunk_offsets:
            if start == end:
                continue

            covered.update(range(start, end))

    uncovered = [
        (i, text[i])
        for i in range(len(text))
        if i not in covered
    ]

    print(f"Nicht abgedeckte Zeichen: {len(uncovered)}")

    non_whitespace = [
        (i, char)
        for i, char in uncovered
        if not char.isspace()
    ]

    print(f"Davon Nicht-Whitespace: {len(non_whitespace)}")

    #if non_whitespace:
    #    print("Beispiele:")
    #    print(non_whitespace[:50])

    return uncovered

In [18]:
check_uncovered_chars(
    fairytale.text,
    encoding["offset_mapping"]
)

Nicht abgedeckte Zeichen: 1325
Davon Nicht-Whitespace: 0


[(2, ' '),
 (6, ' '),
 (12, ' '),
 (20, ' '),
 (23, ' '),
 (27, ' '),
 (36, ' '),
 (41, ' '),
 (50, ' '),
 (55, ' '),
 (61, ' '),
 (65, ' '),
 (72, ' '),
 (79, ' '),
 (87, ' '),
 (93, ' '),
 (98, ' '),
 (105, ' '),
 (110, ' '),
 (114, ' '),
 (122, ' '),
 (126, ' '),
 (129, ' '),
 (136, ' '),
 (140, ' '),
 (144, ' '),
 (150, ' '),
 (158, ' '),
 (162, ' '),
 (167, ' '),
 (170, ' '),
 (177, ' '),
 (185, ' '),
 (190, ' '),
 (195, ' '),
 (207, ' '),
 (210, ' '),
 (214, ' '),
 (218, ' '),
 (222, ' '),
 (226, ' '),
 (234, ' '),
 (242, ' '),
 (247, ' '),
 (251, ' '),
 (255, ' '),
 (264, ' '),
 (268, ' '),
 (275, ' '),
 (279, ' '),
 (283, ' '),
 (290, ' '),
 (298, ' '),
 (304, ' '),
 (308, ' '),
 (311, ' '),
 (315, ' '),
 (321, ' '),
 (327, ' '),
 (333, ' '),
 (339, ' '),
 (345, ' '),
 (349, ' '),
 (353, ' '),
 (362, ' '),
 (367, ' '),
 (371, ' '),
 (375, ' '),
 (379, ' '),
 (385, ' '),
 (390, ' '),
 (395, ' '),
 (398, ' '),
 (403, ' '),
 (407, ' '),
 (418, ' '),
 (425, ' '),
 (428, ' '),
 (432

In [19]:
for fairytale in dataset.fairytales:
    print(f"\n{'='*60}")
    print(f"KHM {fairytale.khm}: {fairytale.title}")
    print(f"{'='*60}")

        # Tokenize
    encoding = tokenizer(
        fairytale.text,
        max_length=512,
        truncation=True,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

    check_uncovered_chars(
    fairytale.text,
    encoding["offset_mapping"]
)


KHM 1: Der Froschkönig oder der eiserne Heinrich
Nicht abgedeckte Zeichen: 1325
Davon Nicht-Whitespace: 0

KHM 2: Katze und Maus in Gesellschaft
Nicht abgedeckte Zeichen: 899
Davon Nicht-Whitespace: 0

KHM 3: Marienkind
Nicht abgedeckte Zeichen: 1791
Davon Nicht-Whitespace: 0

KHM 4: Mährchen von einem, der auszog das Fürchten zu lernen
Nicht abgedeckte Zeichen: 3444
Davon Nicht-Whitespace: 0

KHM 5: Der Wolf und die sieben jungen Geislein
Nicht abgedeckte Zeichen: 1032
Davon Nicht-Whitespace: 0

KHM 6: Der treue Johannes
Nicht abgedeckte Zeichen: 2854
Davon Nicht-Whitespace: 0

KHM 7: Der gute Handel
Nicht abgedeckte Zeichen: 1537
Davon Nicht-Whitespace: 0

KHM 8: Der wunderliche Spielmann
Nicht abgedeckte Zeichen: 996
Davon Nicht-Whitespace: 0

KHM 9: Die zwölf Brüder
Nicht abgedeckte Zeichen: 1860
Davon Nicht-Whitespace: 0

KHM 10: Das Lumpengesindel
Nicht abgedeckte Zeichen: 725
Davon Nicht-Whitespace: 0

KHM 11: Brüderchen und Schwesterchen
Nicht abgedeckte Zeichen: 2296
Davon Ni

## Testing Dataflow

In [20]:
from dataset_preprocessing_transformer import tokenize_fairytale, get_token_label, collate_fn_transformer, RoBERTaSequenceTaggingDataset
from torch.utils.data import DataLoader
from constants import LABEL2ID, ID2LABEL
import torch

In [21]:
all_chunks = []

for fairytale in dataset.fairytales:
    chunks = tokenize_fairytale(
        fairytale,
        tokenizer,
        max_length=512,
        stride=128,
    )
    all_chunks.extend(chunks)

print(f"Anzahl Chunks: {len(all_chunks)}")

Anzahl Chunks: 976


In [22]:
transformer_dataset = RoBERTaSequenceTaggingDataset(
    all_chunks,
    label2id=LABEL2ID
)

In [23]:
transformer_dataloader = DataLoader(
    transformer_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn_transformer,
)

In [24]:
# Checking tensor shape in a batch

batch = next(iter(transformer_dataloader))

print("input_ids:      ", batch["input_ids"].shape)
print("attention_mask: ", batch["attention_mask"].shape)
print("labels:         ", batch["labels"].shape)

ValueError: too many dimensions 'str'

In [ ]:
for i in range(len(batch["input_ids"])):

    valid_tokens = batch["attention_mask"][i].sum().item()
    total_tokens = batch["attention_mask"][i].numel()

    print(
        f"Chunk {i}: "
        f"{valid_tokens}/{total_tokens} echte Tokens"
    )

In [ ]:
i = 1

print(batch["attention_mask"][i])

padding = (
    batch["attention_mask"][i] == 0
).sum().item()

print("Padding:", padding)

# Checking padding -> 42 * 0 for Chunk 1 (470/512 Tokens) is as expected

In [ ]:
i = 1

print(batch["labels"][i])

padding_positions = batch["attention_mask"][i] == 0

print(
    "Labels an Padding-Positionen:",
    batch["labels"][i][padding_positions]
)

assert torch.all(
    batch["labels"][i][padding_positions] == -100
)

-100 as padding for Transformer Learning

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(
    batch["input_ids"][i]
)

for j, token in enumerate(tokens):
    if token in {tokenizer.cls_token, tokenizer.sep_token}:
        print(
            j,
            token,
            batch["labels"][i, j].item()
        )

### Checking Labels on ids
- Sort fairytales by ID
- because: Dataloader shuffle=True

In [25]:
fairytale_by_id = {
    fairytale.khm: fairytale
    for fairytale in dataset.fairytales
}

In [26]:
fairytale = fairytale_by_id[batch["fairytale_id"][0]]

NameError: name 'batch' is not defined

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(
    batch["input_ids"][0]
)

for i, (token, offset) in enumerate(
    zip(tokens, batch["offset_mapping"][0])
):
    label_id = batch["labels"][0, i].item()

    if label_id != LABEL2ID["O"] and label_id != -100:
        start, end = offset

        print(
            f"{i:3} "
            f"{token:20} "
            f"{offset!s:15} "
            f"{ID2LABEL[label_id]:20} "
            f"{fairytale.text[start:end]!r}"
        )

In [ ]:
# Check all Chunks for errors

def check_chunk_offsets(chunks, fairytales):

    fairytale_by_id = {
        fairytale.khm: fairytale
        for fairytale in fairytales
    }

    errors = []

    for chunk in chunks:

        fairytale = fairytale_by_id[chunk["fairytale_id"]]

        for i, (start, end) in enumerate(
            chunk["offset_mapping"]
        ):

            # Special tokens
            if start == end:
                continue

            token_text = fairytale.text[start:end]

            if not token_text:
                errors.append({
                    "khm": chunk["fairytale_id"],
                    "chunk": chunk["chunk_idx"],
                    "token": i,
                    "offset": (start, end),
                })

    return errors

In [ ]:
errors = check_chunk_offsets(
    all_chunks,
    dataset.fairytales
)

print(f"Offset errors: {len(errors)}")

if errors:
    print(errors[:20])

## Testing Chunk Overlap and Labeling

In [27]:
from dataset_preprocessing_transformer import *

In [28]:
def test_chunk_alignment(
    fairytale: Fairytale,
    tokenizer,
    max_length: int = 128,
    stride: int = 32,
):
    """
    Tests tokenization, chunking, offsets and BIO-label alignment
    for one fairytale.

    The test specifically checks whether spans crossing a chunk
    boundary receive B-/I-labels correctly.
    """

    chunks = tokenize_fairytale(
        fairytale=fairytale,
        tokenizer=tokenizer,
        max_length=max_length,
        stride=stride,
    )

    print("=" * 80)
    print(f"Fairytale: {fairytale.khm} - {fairytale.title}")
    print(f"Number of chunks: {len(chunks)}")
    print("=" * 80)

    if len(chunks) < 2:
        print("WARNING: The text does not produce multiple chunks.")
        return

    # ---------------------------------------------------------
    # 1. Check chunk metadata
    # ---------------------------------------------------------

    print("\nCHUNK OVERVIEW")
    print("-" * 80)

    for chunk in chunks:
        offsets = chunk["offset_mapping"]

        valid_offsets = [
            (start, end)
            for start, end in offsets
            if start != end
        ]

        if valid_offsets:
            first_start = valid_offsets[0][0]
            last_end = valid_offsets[-1][1]
        else:
            first_start = None
            last_end = None

        print(
            f"Chunk {chunk['chunk_idx']:>2}: "
            f"sample={chunk['overflow_to_sample_mapping']} | "
            f"chars=({first_start}, {last_end}) | "
            f"tokens={len(chunk['input_ids'])}"
        )

    # ---------------------------------------------------------
    # 2. Check overflow mapping
    # ---------------------------------------------------------

    print("\nOVERFLOW MAPPING")
    print("-" * 80)

    mappings = [
        chunk["overflow_to_sample_mapping"]
        for chunk in chunks
    ]

    print(mappings)

    assert all(mapping == 0 for mapping in mappings), (
        "For a single fairytale all chunks should map to sample 0."
    )

    print("OK: All chunks belong to the same original sample.")

    # ---------------------------------------------------------
    # 3. Check overlap between consecutive chunks
    # ---------------------------------------------------------

    print("\nCHUNK OVERLAPS")
    print("-" * 80)

    overlap_found = False

    for i in range(len(chunks) - 1):

        offsets_a = [
            offset for offset in chunks[i]["offset_mapping"]
            if offset != (0, 0)
        ]

        offsets_b = [
            offset for offset in chunks[i + 1]["offset_mapping"]
            if offset != (0, 0)
        ]

        offsets_a = set(offsets_a)
        offsets_b = set(offsets_b)

        overlap = offsets_a & offsets_b

        if overlap:
            overlap_found = True

            overlap_sorted = sorted(overlap)

            print(
                f"Chunk {i} ↔ Chunk {i+1}: "
                f"{len(overlap_sorted)} overlapping tokens"
            )

            print(
                f"  first overlap: {overlap_sorted[:3]}"
            )

            print(
                f"  last overlap:  {overlap_sorted[-3:]}"
            )

        else:
            print(
                f"Chunk {i} ↔ Chunk {i+1}: NO OVERLAP"
            )

    assert overlap_found, (
        "No overlap detected. Check max_length/stride."
    )

    # ---------------------------------------------------------
    # 4. Print tokens around every chunk boundary
    # ---------------------------------------------------------

    print("\nTOKENS AROUND CHUNK BOUNDARIES")
    print("-" * 80)

    for i in range(len(chunks) - 1):

        chunk_a = chunks[i]
        chunk_b = chunks[i + 1]

        print(f"\nChunk {i} → Chunk {i+1}")

        # Last tokens of chunk A
        print("\n  END OF CHUNK", i)

        tokens_a = tokenizer.convert_ids_to_tokens(
            chunk_a["input_ids"]
        )

        for token, offset, label in zip(
            tokens_a[-5:],
            chunk_a["offset_mapping"][-5:],
            chunk_a["labels"][-5:],
        ):
            print(
                f"    {token:<20} "
                f"{str(offset):<18} "
                f"{label}"
            )

        # First tokens of chunk B
        print("\n  START OF CHUNK", i + 1)

        tokens_b = tokenizer.convert_ids_to_tokens(
            chunk_b["input_ids"]
        )

        for token, offset, label in zip(
            tokens_b[:5],
            chunk_b["offset_mapping"][:5],
            chunk_b["labels"][:5],
        ):
            print(
                f"    {token:<20} "
                f"{str(offset):<18} "
                f"{label}"
            )

    # ---------------------------------------------------------
    # 5. Check spans crossing chunk boundaries
    # ---------------------------------------------------------

    print("\nSPANS CROSSING CHUNK BOUNDARIES")
    print("-" * 80)

    spans = fairytale.resolve_nested_spans()

    crossing_spans = []

    for span in spans:

        for i in range(len(chunks) - 1):

            offsets_a = [
                offset for offset in chunks[i]["offset_mapping"]
                if offset != (0, 0)
            ]

            offsets_b = [
                offset for offset in chunks[i + 1]["offset_mapping"]
                if offset != (0, 0)
            ]

            if not offsets_a or not offsets_b:
                continue

            chunk_a_end = max(end for _, end in offsets_a)
            chunk_b_start = min(start for start, _ in offsets_b)

            # Span crosses the boundary
            if span.start < chunk_a_end and span.end > chunk_b_start:
                crossing_spans.append(
                    (span, i, i + 1)
                )

    if not crossing_spans:
        print("No annotated span crosses a chunk boundary.")

    else:
        for span, chunk_a_idx, chunk_b_idx in crossing_spans:

            print(
                f"\nSpan: {span.text!r}"
            )
            print(
                f"Character offsets: "
                f"({span.start}, {span.end})"
            )
            print(
                f"Label: {span.label}"
            )
            print(
                f"Crosses: "
                f"Chunk {chunk_a_idx} → Chunk {chunk_b_idx}"
            )

            for chunk_idx in [chunk_a_idx, chunk_b_idx]:

                chunk = chunks[chunk_idx]

                tokens = tokenizer.convert_ids_to_tokens(
                    chunk["input_ids"]
                )

                print(f"\n  Chunk {chunk_idx}:")

                for token, offset, label in zip(
                    tokens,
                    chunk["offset_mapping"],
                    chunk["labels"],
                ):

                    token_start, token_end = offset

                    if token_start == token_end:
                        continue

                    # token overlaps annotation
                    if (
                        token_end > span.start
                        and token_start < span.end
                    ):
                        print(
                            f"    {token:<20} "
                            f"{str(offset):<18} "
                            f"{label}"
                        )

    print("\n" + "=" * 80)
    print("TEST FINISHED")
    print("=" * 80)

In [29]:
longest_fairytale = max(
    dataset.fairytales,
    key=lambda x: len(x.text)
)

In [30]:
test_chunk_alignment(
    fairytale=longest_fairytale,
    tokenizer=tokenizer,
    max_length=128,
    stride=32,
)

Fairytale: 60 - Die zwei Brüder
Number of chunks: 126

CHUNK OVERVIEW
--------------------------------------------------------------------------------
Chunk  0: sample=0 | chars=(0, 464) | tokens=128
Chunk  1: sample=0 | chars=(343, 840) | tokens=128
Chunk  2: sample=0 | chars=(714, 1174) | tokens=128
Chunk  3: sample=0 | chars=(1053, 1546) | tokens=128
Chunk  4: sample=0 | chars=(1419, 1912) | tokens=128
Chunk  5: sample=0 | chars=(1782, 2276) | tokens=128
Chunk  6: sample=0 | chars=(2157, 2635) | tokens=128
Chunk  7: sample=0 | chars=(2513, 2979) | tokens=128
Chunk  8: sample=0 | chars=(2854, 3353) | tokens=128
Chunk  9: sample=0 | chars=(3235, 3733) | tokens=128
Chunk 10: sample=0 | chars=(3611, 4107) | tokens=128
Chunk 11: sample=0 | chars=(3973, 4490) | tokens=128
Chunk 12: sample=0 | chars=(4360, 4873) | tokens=128
Chunk 13: sample=0 | chars=(4755, 5226) | tokens=128
Chunk 14: sample=0 | chars=(5109, 5576) | tokens=128
Chunk 15: sample=0 | chars=(5452, 5929) | tokens=128
Chunk 16

### Check aligment

In [31]:
def check_span_token_boundaries(fairytale, tokenizer):
    spans = fairytale.resolve_nested_spans()

    encoding = tokenizer(
        fairytale.text,
        max_length=512,
        stride=128,
        truncation=True,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
        add_special_tokens=True,
    )

    errors = []

    for chunk_idx, offsets in enumerate(encoding["offset_mapping"]):

        for token_idx, (token_start, token_end) in enumerate(offsets):

            # Special token
            if token_start == token_end:
                continue

            for span in spans:

                # Token and span overlap
                if token_end <= span.start or token_start >= span.end:
                    continue

                # Token starts before span and ends inside span
                if token_start < span.start < token_end:
                    errors.append({
                        "chunk": chunk_idx,
                        "token": token_idx,
                        "token_offset": (token_start, token_end),
                        "span_offset": (span.start, span.end),
                        "span_text": span.text,
                        "type": "token starts before span",
                    })

                # Token starts inside span and ends after span
                if token_start < span.end < token_end:
                    errors.append({
                        "chunk": chunk_idx,
                        "token": token_idx,
                        "token_offset": (token_start, token_end),
                        "span_offset": (span.start, span.end),
                        "span_text": span.text,
                        "type": "token ends after span",
                    })

    return errors

In [32]:
all_errors = []

for fairytale in dataset.fairytales:

    errors = check_span_token_boundaries(
        fairytale,
        tokenizer
    )

    for error in errors:
        error["fairytale_id"] = fairytale.khm

    all_errors.extend(errors)

print(f"Gesamtzahl Boundary Errors: {len(all_errors)}")

Gesamtzahl Boundary Errors: 0
